# Week 2 Deliverable: Semantic Search

This notebook demonstrates the semantic search pipeline for the AWS Student Builder Group Week 2 project.

## What is Semantic Search?

Semantic search finds results based on the **meaning** and **intent** of a query, not just exact keyword matches.

## Pipeline Overview
CSV Data → Clean Text → Chunk Text → Generate Embeddings → Store in ChromaDB → Search Query

## Features Demonstrated

- Loading FAQ data from CSV
- Text cleaning (lowercase, punctuation removal)
- Overlapping text chunking
- Semantic embeddings using sentence-transformers
- Vector search using ChromaDB


In [3]:
# Import required libraries
import pandas as pd
import textwrap
import chromadb
from sentence_transformers import SentenceTransformer
from chromadb.utils import embedding_functions

print("Libraries imported successfully!")


Libraries imported successfully!


## Step 1: Load the Data

We use pandas to load FAQ data from a CSV file.


In [4]:
def data_load(path):
    """Load FAQ data from CSV file."""
    df = pd.read_csv(path)
    return df

# Load the FAQ data
df = data_load("data/faq.csv")
print(f"Loaded {len(df)} FAQ entries")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst 3 questions:")
for i, row in df.head(3).iterrows():
    print(f"  {i+1}. {row['question']}")


Loaded 20 FAQ entries

Columns: ['question', 'answer']

First 3 questions:
  1. What is Python and why is it popular for AI?
  2. How does JavaScript differ from Python in web development?
  3. What is React and how does it simplify UI building?


## Step 2: Clean the Text

Text cleaning prepares the data for embedding by:
1. Converting to lowercase
2. Removing punctuation
3. Stripping extra whitespace

This ensures consistent representation and better embeddings.


In [5]:
import re

def clean_text(text):
    """Clean text: lowercase, remove punctuation, strip whitespace."""
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = text.strip()
    return text

# Apply cleaning to questions and answers
df["clean_question"] = df["question"].apply(clean_text)
df["clean_answer"] = df["answer"].apply(clean_text)

print("Cleaned text examples:")
print("\nOriginal: 'What is Python and why is it popular for AI?'")
print(f"Cleaned:  '{df.iloc[0]['clean_question']}'")


Cleaned text examples:

Original: 'What is Python and why is it popular for AI?'
Cleaned:  'what is python and why is it popular for ai'


## Step 3: Chunk the Text

Text chunking splits long text into smaller pieces because:
- Embedding models have context limits
- Smaller chunks = more focused semantic meaning
- Overlapping chunks preserve context at boundaries

Our chunking strategy:
- `chunk_size=35` words per chunk
- `overlap=8` words overlap between chunks

In [6]:
def chunk_text(text, chunk_size=35, overlap=8):
    """Split text into overlapping chunks of words."""
    if chunk_size <= 0:
        raise ValueError("Chunk size must be positive")
    if overlap < 0 or overlap >= chunk_size:
        raise ValueError("overlap must be >= 0 and smaller than chunk size")
    
    words = text.split()
    step = chunk_size - overlap
    return [" ".join(words[start:start + chunk_size]) 
            for start in range(0, len(words), step)]

# Apply chunking to answers
df["chunks"] = df["answer"].apply(lambda x: chunk_text(x))

# Show chunking example
sample_answer = df.iloc[0]["answer"]
sample_chunks = chunk_text(sample_answer)
print(f"Original answer length: {len(sample_answer)} characters")
print(f"Number of chunks: {len(sample_chunks)}")
print(f"\nChunk 1: {sample_chunks[0][:100]}...")
print(f"Chunk 2: {sample_chunks[1][:100]}...")


Original answer length: 241 characters
Number of chunks: 2

Chunk 1: Python is a high-level programming language known for its simplicity and readability. It's popular f...
Chunk 2: TensorFlow, and PyTorch, plus its large community support....


## Step 4: Generate Embeddings

**Embeddings** convert text into numerical vectors (lists of numbers) that capture meaning.

Key concepts:
- Similar texts have similar vectors (close in space)
- We use `all-MiniLM-L6-v2` model (384 dimensions, fast and accurate)
- Cosine similarity measures how similar two vectors are


In [7]:
def get_embeddings(texts):
    """Generate embeddings for a list of texts."""
    model = SentenceTransformer('all-MiniLM-L6-v2')
    embeddings = model.encode(texts, show_progress_bar=True)
    return embeddings

# Generate embeddings for clean questions
print("Generating embeddings for all questions...")
question_texts = list(df["clean_question"])
embeddings = get_embeddings(question_texts)

print(f"\nGenerated {len(embeddings)} embeddings")
print(f"Embedding shape: {embeddings.shape} (texts × dimensions)")
print(f"Each embedding has {embeddings.shape[1]} dimensions")


Generating embeddings for all questions...


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.35it/s]


Generated 20 embeddings
Embedding shape: (20, 384) (texts × dimensions)
Each embedding has 384 dimensions


## Step 5: Vector Search with ChromaDB

**ChromaDB** is a vector database that:
- Stores embeddings efficiently
- Supports fast similarity searches
- Persists data to disk

How it works:
1. Convert query to embedding
2. Find closest vectors in database
3. Return most similar documents


In [8]:
def setup_vector_db():
    """Set up ChromaDB and store all FAQ chunks."""
    embedding_func = embedding_functions.SentenceTransformerEmbeddingFunction(
        model_name="all-MiniLM-L6-v2"
    )
    
    # Create or get existing collection
    client = chromadb.PersistentClient(path="./chroma_db")
    collection = client.get_or_create_collection(
        name="faq_collection",
        embedding_function=embedding_func
    )
    
    # If collection already has data, return it
    if collection.count() > 0:
        print(f"Collection already has {collection.count()} chunks")
        return collection
    
    # Prepare and store chunks
    all_chunks = []
    for idx, row in df.iterrows():
        for i, chunk in enumerate(row["chunks"]):
            all_chunks.append({
                "id": f"q{idx}_chunk{i}",
                "text": chunk,
                "question": row["question"],
                "answer": row["answer"]
            })
    
    # Add to ChromaDB
    collection.add(
        documents=[c["text"] for c in all_chunks],
        ids=[c["id"] for c in all_chunks],
        metadatas=[{"question": c["question"]} for c in all_chunks]
    )
    
    print(f"Stored {len(all_chunks)} chunks in ChromaDB")
    return collection

# Setup the vector database
print("Setting up vector database...")
collection = setup_vector_db()


Setting up vector database...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1393.17it/s]


Collection already has 36 chunks


## Step 6: Search Examples

Now let's search for similar documents using semantic search!

The system will:
1. Convert your query to an embedding
2. Find the closest embeddings in the database
3. Return the most relevant FAQ entries


In [9]:
def search_similar(query, n_results=3):
    """Search for similar documents in vector database."""
    collection = setup_vector_db()
    
    results = collection.query(
        query_texts=[query],
        n_results=n_results
    )
    
    return results

def print_search_results(query, results, max_answer_length=150):
    """Pretty print search results."""
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print(f"{'='*60}")
    
    if not results['documents'] or not results['documents'][0]:
        print("No results found.")
        return
    
    for i, (doc, distance, metadata) in enumerate(zip(
        results['documents'][0],
        results['distances'][0],
        results['metadatas'][0]
    ), 1):
        print(f"\nResult {i} (distance: {distance:.4f})")
        print(f"Question: {metadata['question']}")
        answer = metadata['answer'][:max_answer_length] + "..."
        print(f"Answer: {answer}")
        print(f"{'-'*60}")

# Example searches
print("SEMANTIC SEARCH EXAMPLES")
print("=" * 60)

results = search_similar("How do I install python?")
print_search_results("How do I install python?", results)

results = search_similar("What is the difference between Python and JavaScript?")
print_search_results("What is the difference between Python and JavaScript?", results)

results = search_similar("How does Git work for version control?")
print_search_results("How does Git work for version control?", results)


SEMANTIC SEARCH EXAMPLES
Collection already has 36 chunks

Query: How do I install python?

Result 1 (distance: 0.5264)
Question: What is the purpose of a requirements.txt file?
Answer: requirements.txt lists all Python packages and their specific versions needed for a project. It enables reproducible environments by allowing others t...
------------------------------------------------------------

Result 2 (distance: 0.5586)
Question: What is Python and why is it popular for AI?
Answer: Python is a high-level programming language known for its simplicity and readability. It's popular for AI because of its rich ecosystem of libraries l...
------------------------------------------------------------

Result 3 (distance: 0.6063)
Question: Why is NumPy important for data processing in Python?
Answer: NumPy provides efficient array operations and mathematical functions. It's the foundation for pandas and many ML libraries, enabling fast numerical co...
-------------------------------------

In [10]:
from sentence_transformers import SentenceTransformer
print("SentenceTransformer imported successfully!")


SentenceTransformer imported successfully!
